<a href="https://colab.research.google.com/github/sneyx123-github/CopilotStudioSamples/blob/master/DrStop_Simple_v1_0_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gradio as gr

def gruß_funktion(name):
    return f"Hallo {name}!"

demo = gr.Interface(fn=gruß_funktion, inputs="text", outputs="text")
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a28c604e4d2eb402b9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The code is located in a Google Colab cell.
Can you guide me through deploying this specific code to Google Cloud Run?

The Google Cloud project is project-46810a95-b6e5-47e4-adb.

---


https://share.google/aimode/tNjIcudjj1rYWRxd0

In [ ]:
from google.colab import auth
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"

# Melde dich mit deinem Google-Konto an
auth.authenticate_user()

# Setze das Projekt für die gcloud CLI
!gcloud config set project {PROJECT_ID}


[environment: untagged] Read more g.co/cloud/project-env-tag.
Updated property [core/project].


In [ ]:
!gcloud services enable run.googleapis.com cloudbuild.googleapis.com artifactregistry.googleapis.com


Operation "operations/acat.p2-1074528386995-faf51245-e25a-4a87-8bee-79c88f7bf1d9" finished successfully.


In [ ]:
import os

# Verzeichnis erstellen
os.makedirs('gradio_app', exist_ok=True)

# 1. Die Hauptdatei (app.py) erstellen
with open('gradio_app/main.py', 'w') as f:
    f.write("""
import gradio as gr
import os

def gruß_funktion(name):
    return f"Hallo {name}!"

demo = gr.Interface(fn=gruß_funktion, inputs="text", outputs="text")

if __name__ == "__main__":
    # WICHTIG: Port 8080 für Cloud Run
    demo.launch(server_name="0.0.0.0", server_port=8080)
""")

# 2. Die Abhängigkeiten (requirements.txt) erstellen
with open('gradio_app/requirements.txt', 'w') as f:
    f.write("gradio\n")


In [ ]:
# Wechsel in den Ordner und deploye
%cd gradio_app
!gcloud run deploy gradio-service \
  --source . \
  --region europe-west3 \
  --allow-unauthenticated


/content/SadTalker/gradio_app
Building using Buildpacks and deploying container to Cloud Run service [gradio-service] in project [project-46810a95-b6e5-47e4-adb] region [europe-west3]
Service [gradio-service] revision [gradio-service-00001-xzb] has been deployed and is serving 100 percent of traffic.
Service URL: https://gradio-service-1074528386995.europe-west3.run.app


KORR1


---



In [ ]:
# Projektnummer abfragen
PROJECT_NUMBER = !gcloud projects describe {PROJECT_ID} --format='value(projectNumber)'
PROJECT_NUMBER = PROJECT_NUMBER[0]
print(f"Deine Projektnummer ist: {PROJECT_NUMBER}")


Deine Projektnummer ist: 1074528386995


In [ ]:
BUILD_SA = f"{PROJECT_NUMBER}@cloudbuild.gserviceaccount.com"

# 1. Cloud Run Admin (um den Service zu erstellen/aktualisieren)
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member=serviceAccount:{BUILD_SA} \
    --role=roles/run.admin

# 2. Service Account User (um das Laufzeit-Service-Konto zu verwenden)
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member=serviceAccount:{BUILD_SA} \
    --role=roles/iam.serviceAccountUser

# 3. Storage Object Admin (um auf den Quellcode-Bucket zuzugreifen)
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member=serviceAccount:{BUILD_SA} \
    --role=roles/storage.objectAdmin


Updated IAM policy for project [project-46810a95-b6e5-47e4-adb].
bindings:
- members:
  - serviceAccount:service-1074528386995@gcp-sa-artifactregistry.iam.gserviceaccount.com
  role: roles/artifactregistry.serviceAgent
- members:
  - serviceAccount:service-1074528386995@gcp-sa-cloudaicompanion.iam.gserviceaccount.com
  role: roles/cloudaicompanion.serviceAgent
- members:
  - serviceAccount:1074528386995-compute@developer.gserviceaccount.com
  - serviceAccount:1074528386995@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - serviceAccount:service-1074528386995@gcp-sa-cloudbuild.iam.gserviceaccount.com
  role: roles/cloudbuild.serviceAgent
- members:
  - serviceAccount:service-1074528386995@containerregistry.iam.gserviceaccount.com
  role: roles/containerregistry.ServiceAgent
- members:
  - serviceAccount:1074528386995@cloudbuild.gserviceaccount.com
  role: roles/iam.serviceAccountUser
- members:
  - user:simonborisney@gmail.com
  role: roles/owner
- me

In [ ]:
%cd /content/gradio_app
!gcloud run deploy gradio-service \
  --source . \
  --region europe-west3 \
  --allow-unauthenticated


[Errno 2] No such file or directory: '/content/gradio_app'
/content/SadTalker/gradio_app
Building using Buildpacks and deploying container to Cloud Run service [gradio-service] in project [project-46810a95-b6e5-47e4-adb] region [europe-west3]
Service [gradio-service] revision [gradio-service-00002-ktv] has been deployed and is serving 100 percent of traffic.
Service URL: https://gradio-service-1074528386995.europe-west3.run.app


KORR2


---



In [ ]:
COMPUTE_SA = "1074528386995-compute@developer.gserviceaccount.com"

# Erlaubnis zum Lesen der Quellcode-Dateien im Storage geben
!gcloud projects add-iam-policy-binding project-46810a95-b6e5-47e4-adb \
    --member=serviceAccount:{COMPUTE_SA} \
    --role=roles/storage.objectViewer

# Zusätzlich Sicherheitshalber Cloud Build Rechte geben
!gcloud projects add-iam-policy-binding project-46810a95-b6e5-47e4-adb \
    --member=serviceAccount:{COMPUTE_SA} \
    --role=roles/cloudbuild.builds.builder


Updated IAM policy for project [project-46810a95-b6e5-47e4-adb].
bindings:
- members:
  - serviceAccount:service-1074528386995@gcp-sa-artifactregistry.iam.gserviceaccount.com
  role: roles/artifactregistry.serviceAgent
- members:
  - serviceAccount:service-1074528386995@gcp-sa-cloudaicompanion.iam.gserviceaccount.com
  role: roles/cloudaicompanion.serviceAgent
- members:
  - serviceAccount:1074528386995-compute@developer.gserviceaccount.com
  - serviceAccount:1074528386995@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - serviceAccount:service-1074528386995@gcp-sa-cloudbuild.iam.gserviceaccount.com
  role: roles/cloudbuild.serviceAgent
- members:
  - serviceAccount:service-1074528386995@containerregistry.iam.gserviceaccount.com
  role: roles/containerregistry.ServiceAgent
- members:
  - serviceAccount:1074528386995@cloudbuild.gserviceaccount.com
  role: roles/iam.serviceAccountUser
- members:
  - user:simonborisney@gmail.com
  role: roles/owner
- me

In [ ]:
%cd /content/gradio_app
!gcloud run deploy gradio-service \
  --source . \
  --region europe-west3 \
  --allow-unauthenticated


[Errno 2] No such file or directory: '/content/gradio_app'
/content/SadTalker/gradio_app
Building using Buildpacks and deploying container to Cloud Run service [gradio-service] in project [project-46810a95-b6e5-47e4-adb] region [europe-west3]
Service [gradio-service] revision [gradio-service-00003-vbp] has been deployed and is serving 100 percent of traffic.
Service URL: https://gradio-service-1074528386995.europe-west3.run.app


DNS NAME

---



In [ ]:
# Replace 'your-domain.com' with your actual domain
!gcloud beta run domain-mappings create \
  --service gradio-service \
  --domain us.kommunikator-gmbh.com \
  --region europe-west3




ERROR: (gcloud.beta.run.domain-mappings.create) HttpError accessing <https://europe-west3-run.googleapis.com/apis/domains.cloudrun.com/v1/namespaces/project-46810a95-b6e5-47e4-adb/domainmappings?alt=json>: response: <{'vary': 'Origin, X-Origin, Referer', 'content-type': 'application/json; charset=UTF-8', 'content-encoding': 'gzip', 'date': 'Mon, 06 Apr 2026 11:30:03 GMT', 'server': 'ESF', 'x-xss-protection': '0', 'x-frame-options': 'SAMEORIGIN', 'x-content-type-options': 'nosniff', 'server-timing': 'gfet4t7; dur=1025', 'transfer-encoding': 'chunked', 'status': 501}>, content <{
  "error": {
    "code": 501,
    "message": "Creating domain mappings is not allowed in europe-west3.",
    "status": "UNIMPLEMENTED"
  }
}
>
This may be due to network connectivity issues. Please check your network settings, and the status of the service you are trying to reach.


KORR DNS 1


In [ ]:
%cd /content/gradio_app
!gcloud run deploy gradio-service \
  --source . \
  --region europe-west1 \
  --allow-unauthenticated


[Errno 2] No such file or directory: '/content/gradio_app'
/content/SadTalker/gradio_app
Building using Buildpacks and deploying container to Cloud Run service [gradio-service] in project [project-46810a95-b6e5-47e4-adb] region [europe-west1]
Service [gradio-service] revision [gradio-service-00005-pnt] has been deployed and is serving 100 percent of traffic.
Service URL: https://gradio-service-1074528386995.europe-west1.run.app


In [ ]:
!gcloud beta run domain-mappings create \
  --service gradio-service \
  --domain us.kommunikator-gmbh.com \
  --region europe-west1


ERROR: (gcloud.beta.run.domain-mappings.create) Domain mapping to [us.kommunikator-gmbh.com] already exists in this region.


In [ ]:
# Replace 'your-domain.com' with your actual domain
!gcloud beta run domain-mappings describe \
  --domain us.kommunikator-gmbh.com \
  --region europe-west1 \
  --format="yaml(status)"


status:
  conditions:
  - lastTransitionTime: '2026-04-05T15:24:49.463975Z'
    message: Waiting for certificate provisioning. You must configure your DNS records
      for certificate issuance to begin.
    reason: CertificatePending
    status: Unknown
    type: Ready
  - lastTransitionTime: '2026-04-05T16:01:13.479104Z'
    message: Certificate issuance pending. The challenge data was not visible through
      the public internet. This may indicate that DNS is not properly configured or
      has not fully propagated. The system will retry.
    reason: CertificatePending
    status: Unknown
    type: CertificateProvisioned
  - lastTransitionTime: '2026-04-05T15:24:49.463975Z'
    status: 'True'
    type: DomainRoutable
  - lastTransitionTime: '2026-04-06T10:51:21.490359Z'
    message: System will retry after 24:00:00 from lastTransitionTime for polling
      interval
    reason: WaitingForOperation
    severity: Info
    status: 'True'
    type: Retry
  mappedRouteName: gradio-servi

ADD PW


---



In [ ]:
%cd /content

import os

# Update the main.py file with authentication
with open('gradio_app/main.py', 'w') as f:
    f.write("""
import gradio as gr
import os

def gruß_funktion(name):
    return f"Hallo {name}!"

demo = gr.Interface(fn=gruß_funktion, inputs="text", outputs="text")

if __name__ == "__main__":
    # Add authentication: (username, password)
    # You can also use a list for multiple users: [("admin", "pass123"), ("user", "user456")]
    demo.launch(
        server_name="0.0.0.0",
        server_port=8080,
        auth=("admin", "dein_geheimes_passwort")
    )
""")


/content


FileNotFoundError: [Errno 2] No such file or directory: 'gradio_app/main.py'

In [ ]:
%cd /content/gradio_app
!gcloud run deploy gradio-service \
  --source . \
  --region europe-west1 \
  --allow-unauthenticated


[Errno 2] No such file or directory: '/content/gradio_app'
/content
Building using Buildpacks and deploying container to Cloud Run service [gradio-service] in project [project-46810a95-b6e5-47e4-adb] region [europe-west1]
ERROR: (gcloud.run.deploy) [Errno 95] Operation not supported: 'drive/MyDrive/ORDERS_APO.gsheet'
This may be due to network connectivity issues. Please check your network settings, and the status of the service you are trying to reach.


In [ ]:
# 1. Enable the Secret Manager API
!gcloud services enable ://googleapis.com

# 2. Create the secret container
!gcloud secrets create GRADIO_PASSWORD --replication-policy="automatic"

# 3. Add the actual password to the secret
# Replace 'mein_geheimes_passwort' with your chosen password
!echo -n "mein_geheimes_passwort" | gcloud secrets versions add GRADIO_PASSWORD --data-file=-


In [ ]:
# Get your project number again
PROJECT_NUMBER = !gcloud projects describe {PROJECT_ID} --format='value(projectNumber)'
COMPUTE_SA = f"{PROJECT_NUMBER[0]}-compute@://gserviceaccount.com"

# Give the service account access to read secrets
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member=serviceAccount:{COMPUTE_SA} \
    --role=roles/secretmanager.secretAccessor


In [ ]:
# Fixed Service Account string
COMPUTE_SA = "1074528386995-compute@developer.gserviceaccount.com"
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"

# Grant the Secret Manager accessor role
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member=serviceAccount:{COMPUTE_SA} \
    --role=roles/secretmanager.secretAccessor


In [ ]:
%cd /content/gradio_app
!gcloud run deploy gradio-service \
  --source . \
  --region europe-west1 \
  --set-secrets="MY_APP_PASSWORD=GRADIO_PASSWORD:latest" \
  --allow-unauthenticated


In [ ]:
# 3. Add the actual password to the secret
# Replace 'mein_geheimes_passwort' with your chosen password
!echo -n "mein_geheimes_passwort" | gcloud secrets versions add GRADIO_PASSWORD --data-file=-

In [ ]:
%cd /content

with open('gradio_app/main.py', 'w') as f:
    f.write("""
import gradio as gr
import os

# Function to get secret from environment (injected by Cloud Run)
def get_password():
    return os.environ.get("MY_APP_PASSWORD", "default_fallback_pass")

def gruß_funktion(name):
    return f"Hallo {name}!"

demo = gr.Interface(fn=gruß_funktion, inputs="text", outputs="text")

if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",
        server_port=8080,
        auth=("admin", get_password())
    )
""")


In [ ]:
%cd /content/gradio_app
!gcloud run deploy gradio-service \
  --source . \
  --region europe-west1 \
  --set-secrets="MY_APP_PASSWORD=GRADIO_PASSWORD:latest" \
  --allow-unauthenticated


In [ ]:
# Replace with your actual domain
!host gradio-service.us.kommunikator-gmbh.com


In [ ]:
!gcloud beta run domain-mappings describe \
  --domain us.kommunikator-gmbh.com \
  --region europe-west1 \
  --format="yaml(status)"


What is required to stop and remove the gradio-service for europe-west3 without affecting the service in europe-west1?


---



In [ ]:
!gcloud run services delete gradio-service --region europe-west3 --quiet
